In [ ]:
# Fabric Notebook: NB_20_SILVER_POI_COMPONENTS
# READ/WRITE: Silver only
# Uses _stage_poi_language as the canonical incremental input.

from pyspark.sql import functions as F

# NB20 EXECUTION CONTRACT
# Default Lakehouse MUST be lh_vv_silver.
# All Silver source/target tables intentionally use single-part names.
# Do not change them back to lh_vv_silver.dbo.* in this notebook.
run_id = globals().get("run_id", "DEV_RUN")
STAGE = "_stage_poi_language"

# Namespace/schema preflight. spark.table(single_part) must resolve against the
# notebook's default Lakehouse (lh_vv_silver).
_required_silver_tables = [
    STAGE,
    "poi_address",
    "poi_contact",
    "poi_price",
    "poi_content",
    "poi_rating",
    "poi_review",
    "poi_media",
    "poi_opening_hours",
    "poi_policy",
    "poi_amenity_source",
    "poi_amenity_current",
]
for _tbl in _required_silver_tables:
    try:
        spark.table(_tbl).limit(0).collect()
    except Exception as _e:
        raise RuntimeError(
            f"NB20_SILVER_TABLE_RESOLUTION_FAILED: table={_tbl}. "
            "Attach/default this notebook to Lakehouse lh_vv_silver. "
            f"Original error: {_e}"
        ) from _e

print("NB20 Silver namespace preflight = OK (single-part tables, default Lakehouse lh_vv_silver)")

# Register any existing mapping UDFs from V1 here unchanged if they are still required:
#   fabric_map_sector(primaryType, types_json)
#   fabric_map_category(primaryType, types_json)
#   fabric_build_amenities_from_raw(sector, category, raw_data_json)
# This version keeps business mapping semantics unchanged; only the multilingual input stage changes.

# -----------------------------------------------------------------------------
# 1. poi_address - scalar, latest per poi_id
# -----------------------------------------------------------------------------
spark.sql(f"""
MERGE INTO poi_address tgt
USING (
  SELECT *, current_timestamp() _now FROM (
    SELECT
      poi_id,
      get_json_object(address_json,'$.address_line1') address_line1,
      get_json_object(address_json,'$.address_line2') address_line2,
      get_json_object(address_json,'$.ward') ward,
      get_json_object(address_json,'$.district') district,
      get_json_object(address_json,'$.city') city,
      get_json_object(address_json,'$.state_province') state_province,
      get_json_object(address_json,'$.country_code') country_code,
      get_json_object(address_json,'$.postal_code') postal_code,
      get_json_object(address_json,'$.short_address') short_address,
      get_json_object(address_json,'$.full_address') full_address,
      get_json_object(address_json,'$.address_viet_map') address_viet_map,
      CAST(get_json_object(address_json,'$.lat') AS DOUBLE) lat,
      CAST(get_json_object(address_json,'$.lng') AS DOUBLE) lng,
      get_json_object(address_json,'$.geohash') geohash,
      get_json_object(address_json,'$.h3_index_r8') h3_res8,
      get_json_object(address_json,'$.plus_code') plus_code,
      CAST(NULL AS DOUBLE) entrance_lat,
      CAST(NULL AS DOUBLE) entrance_lng,
      event_id _batch_id
    FROM {STAGE}
    WHERE run_id='{run_id}' AND rn_by_poi=1
  )
) src
ON tgt.poi_id=src.poi_id
WHEN MATCHED THEN UPDATE SET
  tgt.address_line1=src.address_line1, tgt.address_line2=src.address_line2,
  tgt.ward=src.ward, tgt.district=src.district, tgt.city=src.city,
  tgt.state_province=src.state_province, tgt.country_code=src.country_code,
  tgt.postal_code=src.postal_code, tgt.short_address=src.short_address,
  tgt.full_address=src.full_address, tgt.address_viet_map=src.address_viet_map,
  tgt.lat=src.lat, tgt.lng=src.lng, tgt.geohash=src.geohash, tgt.h3_res8=src.h3_res8,
  tgt.plus_code=src.plus_code, tgt.entrance_lat=src.entrance_lat, tgt.entrance_lng=src.entrance_lng,
  tgt._batch_id=src._batch_id, tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT (
  poi_id,address_line1,address_line2,ward,district,city,state_province,country_code,postal_code,
  short_address,full_address,address_viet_map,lat,lng,geohash,h3_res8,plus_code,entrance_lat,entrance_lng,
  _batch_id,_created_at,_updated_at
) VALUES (
  src.poi_id,src.address_line1,src.address_line2,src.ward,src.district,src.city,src.state_province,
  src.country_code,src.postal_code,src.short_address,src.full_address,src.address_viet_map,
  src.lat,src.lng,src.geohash,src.h3_res8,src.plus_code,src.entrance_lat,src.entrance_lng,
  src._batch_id,src._now,src._now
)
""")

# -----------------------------------------------------------------------------
# 2. poi_contact - preserve V1 semantics: no update if contact block absent.
# -----------------------------------------------------------------------------
spark.sql(f"""
MERGE INTO poi_contact tgt
USING (
 SELECT *,current_timestamp() _now FROM (
  SELECT
    poi_id,
    get_json_object(contact_json,'$.phone') phone,
    CASE WHEN get_json_object(contact_json,'$.hotline') IS NOT NULL
         THEN array(get_json_object(contact_json,'$.hotline'))
         ELSE CAST(NULL AS ARRAY<STRING>) END secondary_phones,
    get_json_object(contact_json,'$.email') email,
    get_json_object(contact_json,'$.website_url') website_url,
    COALESCE(
      get_json_object(contact_json,'$.booking_links.agoda'),
      get_json_object(contact_json,'$.booking_links.booking_com'),
      get_json_object(contact_json,'$.booking_links.traveloka'),
      get_json_object(contact_json,'$.booking_links.airbnb'),
      get_json_object(contact_json,'$.booking_links.vntrip')
    ) booking_url,
    CAST(NULL AS STRING) menu_url,
    get_json_object(contact_json,'$.social_links.facebook') facebook_url,
    get_json_object(contact_json,'$.social_links.instagram') instagram_url,
    get_json_object(contact_json,'$.social_links.tiktok') tiktok_url,
    CAST(NULL AS STRING) whatsapp_number,
    event_id _batch_id
  FROM {STAGE}
  WHERE run_id='{run_id}' AND rn_by_poi=1 AND contact_json IS NOT NULL
 )
) src
ON tgt.poi_id=src.poi_id
WHEN MATCHED THEN UPDATE SET
 tgt.phone=src.phone,tgt.secondary_phones=src.secondary_phones,tgt.email=src.email,
 tgt.website_url=src.website_url,tgt.booking_url=src.booking_url,tgt.menu_url=src.menu_url,
 tgt.facebook_url=src.facebook_url,tgt.instagram_url=src.instagram_url,tgt.tiktok_url=src.tiktok_url,
 tgt.whatsapp_number=src.whatsapp_number,tgt._batch_id=src._batch_id,tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT (
 poi_id,phone,secondary_phones,email,website_url,booking_url,menu_url,facebook_url,instagram_url,tiktok_url,
 whatsapp_number,_batch_id,_created_at,_updated_at
) VALUES (
 src.poi_id,src.phone,src.secondary_phones,src.email,src.website_url,src.booking_url,src.menu_url,
 src.facebook_url,src.instagram_url,src.tiktok_url,src.whatsapp_number,src._batch_id,src._now,src._now
)
""")

# -----------------------------------------------------------------------------
# 3. poi_price
# -----------------------------------------------------------------------------
spark.sql(f"""
MERGE INTO poi_price tgt
USING (
 SELECT *,current_timestamp() _now FROM (
  SELECT poi_id,
    COALESCE(get_json_object(price_json,'$.currency'),get_json_object(raw_data_json,'$.priceRange.startPrice.currencyCode')) currency,
    COALESCE(get_json_object(price_json,'$.price_level'),get_json_object(raw_data_json,'$.priceLevel')) price_level,
    COALESCE(CAST(get_json_object(price_json,'$.price_min') AS DOUBLE),CAST(get_json_object(raw_data_json,'$.priceRange.startPrice.units') AS DOUBLE)) price_min,
    COALESCE(CAST(get_json_object(price_json,'$.price_max') AS DOUBLE),CAST(get_json_object(raw_data_json,'$.priceRange.endPrice.units') AS DOUBLE)) price_max,
    event_id _batch_id
  FROM {STAGE}
  WHERE run_id='{run_id}' AND rn_by_poi=1 AND (price_json IS NOT NULL OR raw_data_json IS NOT NULL)
 )
) src
ON tgt.poi_id=src.poi_id
WHEN MATCHED THEN UPDATE SET tgt.currency=src.currency,tgt.price_level=src.price_level,tgt.price_min=src.price_min,tgt.price_max=src.price_max,tgt._batch_id=src._batch_id,tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT (poi_id,currency,price_level,price_min,price_max,_batch_id,_created_at,_updated_at)
VALUES(src.poi_id,src.currency,src.price_level,src.price_min,src.price_max,src._batch_id,src._now,src._now)
""")

# -----------------------------------------------------------------------------
# 4. poi_content - only content matching the logical POI language is selected first.
# If no matching locale exists, preserve V1 fallback behavior by keeping all locale rows.
# -----------------------------------------------------------------------------
spark.sql(f"""
MERGE INTO poi_content tgt
USING (
 WITH exploded AS (
   SELECT s.poi_id,s.lang,s.event_id,s.crawled_at,c.*
   FROM {STAGE} s
   LATERAL VIEW explode(from_json(s.content_json,
     'ARRAY<STRUCT<content_id:STRING,locale:STRING,seo_title:STRING,seo_description:STRING,text:STRING,service_bio:STRING,service_story:STRING,service_usps:ARRAY<STRING>,keywords:ARRAY<STRING>,local_tips:STRING,important_notice:STRING,is_original:BOOLEAN,translation_status:STRING,created_at:STRING>>')) x AS c
   WHERE s.run_id='{run_id}' AND s.content_json IS NOT NULL
 ), ranked AS (
   SELECT *, ROW_NUMBER() OVER(PARTITION BY poi_id,COALESCE(locale,lang) ORDER BY crawled_at DESC) rn
   FROM exploded
 )
 SELECT
   poi_id,COALESCE(locale,lang) locale,
   NULL AS _dummy,
   current_timestamp() _now,
   event_id _batch_id,
   seo_title display_name_override,
   text,seo_description,service_story,service_bio,service_usps,important_notice,keywords,created_at
 FROM ranked WHERE rn=1
) src
ON tgt.poi_id=src.poi_id AND tgt.locale=src.locale
WHEN MATCHED THEN UPDATE SET
 tgt.display_name=COALESCE(tgt.display_name,src.display_name_override),
 tgt.description_short=COALESCE(src.text,src.seo_description),
 tgt.description_long=src.service_story,tgt.service_bio=src.service_bio,tgt.service_usps=src.service_usps,
 tgt.important_notice=src.important_notice,tgt.tags_text=src.keywords,
 tgt.seo_title=src.display_name_override,tgt.seo_description=src.seo_description,
 tgt.content_updated_at=CAST(src.created_at AS TIMESTAMP),tgt._batch_id=src._batch_id,tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT(
 poi_id,locale,display_name,description_short,description_long,service_bio,service_usps,important_notice,tags_text,
 price_range_text,hours_weekday_text,seo_title,seo_description,content_updated_at,_batch_id,_created_at,_updated_at
) VALUES(
 src.poi_id,src.locale,src.display_name_override,COALESCE(src.text,src.seo_description),src.service_story,src.service_bio,
 src.service_usps,src.important_notice,src.keywords,NULL,NULL,src.display_name_override,src.seo_description,
 CAST(src.created_at AS TIMESTAMP),src._batch_id,src._now,src._now
)
""")

# -----------------------------------------------------------------------------
# 5. poi_rating
# -----------------------------------------------------------------------------
spark.sql(f"""
MERGE INTO poi_rating tgt
USING (
 SELECT *,current_timestamp() _now FROM (
  SELECT poi_id,
   CAST(get_json_object(rating_json,'$.rating_overall') AS DECIMAL(3,1)) overall,
   COALESCE(CAST(get_json_object(rating_json,'$.rating_count') AS BIGINT),0) total_reviews,
   get_json_object(rating_json,'$.rating_label') score_label,
   CAST(get_json_object(rating_json,'$.rating_weighted') AS DECIMAL(4,2)) rating_weighted,
   to_json(array(named_struct('source',get_json_object(rating_json,'$.source'),'rating_overall',CAST(get_json_object(rating_json,'$.rating_overall') AS DOUBLE),'rating_count',CAST(get_json_object(rating_json,'$.rating_count') AS BIGINT)))) sources_json,
   rating_json breakdown_json,
   CAST(NULL AS DECIMAL(4,3)) sentiment_avg,
   CAST(get_json_object(rating_json,'$.last_review_at') AS TIMESTAMP) last_review_at,
   event_id _batch_id
  FROM {STAGE} WHERE run_id='{run_id}' AND rn_by_poi=1 AND rating_json IS NOT NULL
 )
) src
ON tgt.poi_id=src.poi_id
WHEN MATCHED THEN UPDATE SET tgt.overall=src.overall,tgt.total_reviews=src.total_reviews,tgt.score_label=src.score_label,tgt.rating_weighted=src.rating_weighted,tgt.sources_json=src.sources_json,tgt.breakdown_json=src.breakdown_json,tgt.sentiment_avg=src.sentiment_avg,tgt.last_review_at=src.last_review_at,tgt._batch_id=src._batch_id,tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT(poi_id,overall,total_reviews,score_label,rating_weighted,sources_json,breakdown_json,sentiment_avg,last_review_at,_batch_id,_created_at,_updated_at)
VALUES(src.poi_id,src.overall,src.total_reviews,src.score_label,src.rating_weighted,src.sources_json,src.breakdown_json,src.sentiment_avg,src.last_review_at,src._batch_id,src._now,src._now)
""")

# -----------------------------------------------------------------------------
# 6. poi_review - IMPORTANT V2 change: review_id includes poi_id so shared root reviews
# can be replicated safely to N language-specific POIs.
# -----------------------------------------------------------------------------
spark.sql(f"""
MERGE INTO poi_review tgt
USING (
 SELECT *,current_timestamp() _now FROM (
  SELECT
   s.poi_id,s.source_name,
   sha2(concat(s.source_name,s.source_id,COALESCE(r.author_name,''),COALESCE(r.time,'')),256) source_review_id,
   sha2(concat(s.poi_id,'|',s.source_name,'|',s.source_id,'|',COALESCE(r.author_name,''),'|',COALESCE(r.time,'')),256) review_id,
   s.lang language,r.author_name reviewer_name,CAST(NULL AS STRING) reviewer_id_hash,
   CAST(r.rating AS DECIMAL(3,1)) rating_score,CAST(NULL AS STRING) review_title,r.text review_text,
   CAST(NULL AS ARRAY<STRING>) review_media_urls,CAST(r.time AS DATE) review_date,
   CAST(NULL AS STRING) travel_type,CAST(NULL AS BIGINT) helpful_count,
   CAST(NULL AS DOUBLE) fraud_score,CAST(NULL AS ARRAY<STRING>) fraud_reasons,true is_visible,
   CAST(NULL AS STRING) response_text,CAST(NULL AS TIMESTAMP) responded_at,CAST(r.time AS TIMESTAMP) published_at,
   s.event_id _batch_id,s.crawled_at,
   ROW_NUMBER() OVER(PARTITION BY s.poi_id,sha2(concat(s.source_name,s.source_id,COALESCE(r.author_name,''),COALESCE(r.time,'')),256) ORDER BY s.crawled_at DESC) rn
  FROM {STAGE} s
  LATERAL VIEW explode(from_json(s.review_json,'ARRAY<STRUCT<author_name:STRING,rating:INT,source:STRING,text:STRING,time:STRING>>')) x AS r
  WHERE s.run_id='{run_id}' AND s.review_json IS NOT NULL AND r.author_name IS NOT NULL AND r.time IS NOT NULL
 ) WHERE rn=1
) src
ON tgt.review_id=src.review_id
WHEN MATCHED THEN UPDATE SET tgt.poi_id=src.poi_id,tgt.is_visible=src.is_visible,tgt.language=src.language,tgt.helpful_count=src.helpful_count,tgt.fraud_score=src.fraud_score,tgt.fraud_reasons=src.fraud_reasons,tgt.response_text=src.response_text,tgt.responded_at=src.responded_at,tgt._batch_id=src._batch_id,tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT(poi_id,source_name,source_review_id,review_id,language,reviewer_name,reviewer_id_hash,rating_score,review_title,review_text,review_media_urls,review_date,travel_type,helpful_count,fraud_score,fraud_reasons,is_visible,response_text,responded_at,published_at,_batch_id,_created_at,_updated_at)
VALUES(src.poi_id,src.source_name,src.source_review_id,src.review_id,src.language,src.reviewer_name,src.reviewer_id_hash,src.rating_score,src.review_title,src.review_text,src.review_media_urls,src.review_date,src.travel_type,src.helpful_count,src.fraud_score,src.fraud_reasons,src.is_visible,src.response_text,src.responded_at,src.published_at,src._batch_id,src._now,src._now)
""")

# -----------------------------------------------------------------------------
# 7. poi_media - cumulative dedup by Data Collector deterministic media_id (V7)
#
# AUTHORITATIVE IDENTITY:
#   source_name + source_id + media_id
# media_id is deterministic/hash-based in Data Collector: same image => same media_id.
# photo_api_uri remains metadata only; it is NOT part of dedup identity.
# Silver row identity remains POI-scoped via poi_id + media_dedup_key.
# -----------------------------------------------------------------------------
MEDIA_TABLE = "poi_media"

for _col_def in ["photo_api_uri STRING", "media_dedup_key STRING"]:
    try:
        spark.sql(f"ALTER TABLE {MEDIA_TABLE} ADD COLUMNS ({_col_def})")
    except Exception:
        pass

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v_nb20_media_current AS
WITH exploded AS (
    SELECT s.poi_id,s.source_name,s.source_id,s.event_id AS _batch_id,s.crawled_at,m.*,
           SHA2(CONCAT_WS('|',TRIM(s.source_name),TRIM(s.source_id),TRIM(m.media_id)),256)
             AS canonical_media_dedup_key
    FROM {STAGE} s
    LATERAL VIEW explode(from_json(
      s.media_json,
      'ARRAY<STRUCT<media_id:STRING,media_fingerprint:STRING,photo_api_uri:STRING,media_type:STRING,blob_url:STRING,original_url:STRING,thumbnail_url:STRING,processing_status:STRING,category:STRING,is_blessed:BOOLEAN,display_order:INT,photographer:STRING,license:STRING,source_field:STRING,source:STRING,width_px:INT,height_px:INT,created_at:STRING,updated_at:STRING>>'
    )) x AS m
    WHERE s.run_id='{run_id}' AND s.media_json IS NOT NULL
      AND m.media_id IS NOT NULL AND TRIM(m.media_id)<>''
), ranked AS (
    SELECT *,ROW_NUMBER() OVER(
      PARTITION BY poi_id,source_name,source_id,media_id
      ORDER BY crawled_at ASC,COALESCE(display_order,999999),media_id
    ) rn
    FROM exploded
)
SELECT * FROM ranked WHERE rn=1
""")

# Migrate historical rows to the new deterministic identity when their stored
# source fields are complete. source_media_id stores the POI source_id in this schema.
spark.sql(f"""
UPDATE {MEDIA_TABLE}
SET media_dedup_key=SHA2(CONCAT_WS('|',
    TRIM(source_name),TRIM(source_media_id),TRIM(media_id)),256)
WHERE source_name IS NOT NULL AND TRIM(source_name)<>''
  AND source_media_id IS NOT NULL AND TRIM(source_media_id)<>''
  AND media_id IS NOT NULL AND TRIM(media_id)<>''
""")

# Current crawl is authoritative for an existing poi_id + media_id and migrates
# any row created under the previous photo_api_uri-based contract.
spark.sql(f"""
MERGE INTO {MEDIA_TABLE} tgt
USING v_nb20_media_current src
ON tgt.poi_id=src.poi_id AND tgt.media_id=src.media_id
WHEN MATCHED THEN UPDATE SET
  tgt.media_dedup_key=src.canonical_media_dedup_key,
  tgt.source_name=COALESCE(tgt.source_name,src.source_name),
  tgt.source_media_id=COALESCE(tgt.source_media_id,src.source_id),
  tgt.photo_api_uri=COALESCE(tgt.photo_api_uri,src.photo_api_uri),
  tgt._updated_at=current_timestamp()
""")

# Fabric-safe historical duplicate cleanup: no correlated DELETE subquery.
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v_nb20_media_duplicates AS
WITH ranked AS (
  SELECT poi_id,media_id,media_dedup_key,blob_url,_created_at,_updated_at,
         ROW_NUMBER() OVER(
           PARTITION BY poi_id,media_dedup_key
           ORDER BY _created_at ASC NULLS LAST,_updated_at ASC NULLS LAST,
                    media_id ASC,blob_url ASC NULLS LAST
         ) rn
  FROM {MEDIA_TABLE}
  WHERE media_dedup_key IS NOT NULL AND TRIM(media_dedup_key)<>''
)
SELECT poi_id,media_id,media_dedup_key,blob_url,_created_at,_updated_at
FROM ranked WHERE rn>1
""")

spark.sql(f"""
MERGE INTO {MEDIA_TABLE} tgt
USING v_nb20_media_duplicates src
ON tgt.poi_id=src.poi_id
AND tgt.media_id=src.media_id
AND tgt.media_dedup_key=src.media_dedup_key
AND COALESCE(tgt.blob_url,'')=COALESCE(src.blob_url,'')
AND COALESCE(CAST(tgt._created_at AS STRING),'')=COALESCE(CAST(src._created_at AS STRING),'')
AND COALESCE(CAST(tgt._updated_at AS STRING),'')=COALESCE(CAST(src._updated_at AS STRING),'')
WHEN MATCHED THEN DELETE
""")

spark.sql(f"""
MERGE INTO {MEDIA_TABLE} tgt
USING (SELECT *,current_timestamp() _now FROM v_nb20_media_current) src
ON tgt.poi_id=src.poi_id AND tgt.media_dedup_key=src.canonical_media_dedup_key
WHEN MATCHED THEN UPDATE SET
  tgt.photo_api_uri=COALESCE(tgt.photo_api_uri,src.photo_api_uri),
  tgt.processing_status=COALESCE(src.processing_status,tgt.processing_status),
  tgt.is_blessed=COALESCE(src.is_blessed,tgt.is_blessed,false),
  tgt.display_order=COALESCE(tgt.display_order,src.display_order),
  tgt.thumbnail_url=COALESCE(tgt.thumbnail_url,src.thumbnail_url),
  tgt.is_active=true,
  tgt.source_updated_at=GREATEST(tgt.source_updated_at,CAST(src.updated_at AS TIMESTAMP)),
  tgt._batch_id=src._batch_id,tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT(
  poi_id,media_id,media_dedup_key,photo_api_uri,source_name,source_media_id,
  media_type,blob_url,original_url,thumbnail_url,processing_status,category,is_blessed,
  display_order,photographer,license,source_field,width_px,height_px,caption,
  source_created_at,source_updated_at,is_active,_batch_id,_created_at,_updated_at
) VALUES(
  src.poi_id,src.media_id,src.canonical_media_dedup_key,src.photo_api_uri,
  src.source_name,src.source_id,src.media_type,src.blob_url,src.original_url,
  src.thumbnail_url,src.processing_status,src.category,COALESCE(src.is_blessed,false),
  src.display_order,src.photographer,src.license,src.source_field,src.width_px,
  src.height_px,NULL,CAST(src.created_at AS TIMESTAMP),CAST(src.updated_at AS TIMESTAMP),
  true,src._batch_id,src._now,src._now
)
""")

_media_metrics=spark.sql(f"""
WITH p AS (SELECT DISTINCT poi_id FROM v_nb20_media_current),
t AS (SELECT m.* FROM {MEDIA_TABLE} m JOIN p ON p.poi_id=m.poi_id),
d AS (
 SELECT poi_id,media_dedup_key,COUNT(*) c FROM t
 WHERE media_dedup_key IS NOT NULL AND TRIM(media_dedup_key)<>''
 GROUP BY poi_id,media_dedup_key HAVING COUNT(*)>1
)
SELECT (SELECT COUNT(*) FROM v_nb20_media_current) media_current_unique_count,
       (SELECT COUNT(*) FROM t) media_target_row_count,
       (SELECT COUNT(*) FROM d) duplicate_media_key_groups
""").first()
print("NB20 media V7:",dict(_media_metrics.asDict()),
      "identity=poi_id + SHA256(source_name|source_id|media_id)")
if (_media_metrics["duplicate_media_key_groups"] or 0)>0:
    raise RuntimeError("NB20_MEDIA_INVARIANT_FAILED: duplicate poi_id + media_dedup_key remains")

# -----------------------------------------------------------------------------
# 8. opening_hours - keep current V1 persisted semantics for this version.
# -----------------------------------------------------------------------------
spark.sql(f"""
MERGE INTO poi_opening_hours tgt
USING (
 SELECT poi_id,
   get_json_object(address_json,'$.timezone') timezone,
   COALESCE(CAST(get_json_object(opening_hours_json,'$.is_24_7') AS BOOLEAN),false) is_24_7,
   CAST(NULL AS BOOLEAN) open_now,
   get_json_object(opening_hours_json,'$.periods') periods_json,
   get_json_object(opening_hours_json,'$.holiday_exceptions') holiday_exceptions_json,
   CAST(get_json_object(opening_hours_json,'$.created_at') AS TIMESTAMP) source_last_changed_at,
   event_id _batch_id,current_timestamp() _now
 FROM {STAGE}
 WHERE run_id='{run_id}' AND rn_by_poi=1 AND opening_hours_json IS NOT NULL
) src
ON tgt.poi_id=src.poi_id
WHEN MATCHED THEN UPDATE SET tgt.timezone=src.timezone,tgt.is_24_7=src.is_24_7,tgt.open_now=src.open_now,tgt.periods_json=src.periods_json,tgt.holiday_exceptions_json=src.holiday_exceptions_json,tgt.source_last_changed_at=src.source_last_changed_at,tgt._batch_id=src._batch_id,tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT(poi_id,timezone,is_24_7,open_now,periods_json,holiday_exceptions_json,source_last_changed_at,_batch_id,_created_at,_updated_at)
VALUES(src.poi_id,src.timezone,src.is_24_7,src.open_now,src.periods_json,src.holiday_exceptions_json,src.source_last_changed_at,src._batch_id,src._now,src._now)
""")

# -----------------------------------------------------------------------------
# 9. poi_policy
# -----------------------------------------------------------------------------
spark.sql(f"""
MERGE INTO poi_policy tgt
USING (
 SELECT poi_id,policy_json policy_json,CAST(NULL AS TIMESTAMP) source_updated_at,event_id _batch_id,current_timestamp() _now
 FROM {STAGE} WHERE run_id='{run_id}' AND rn_by_poi=1 AND policy_json IS NOT NULL
) src ON tgt.poi_id=src.poi_id
WHEN MATCHED THEN UPDATE SET tgt.policy_json=src.policy_json,tgt.source_updated_at=src.source_updated_at,tgt._batch_id=src._batch_id,tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT(poi_id,policy_json,source_updated_at,_batch_id,_created_at,_updated_at)
VALUES(src.poi_id,src.policy_json,src.source_updated_at,src._batch_id,src._now,src._now)
""")

# -----------------------------------------------------------------------------
# 10/11. Amenity source/current.
# For V2 phase 1, raw amenity is replicated to each language-specific poi_id.
# Mapping expressions can be replaced by the existing V1 mapping UDF outputs.
# -----------------------------------------------------------------------------
spark.sql(f"""
MERGE INTO poi_amenity_source tgt
USING (
 SELECT poi_id,source_name,source_id source_poi_id,
   COALESCE(business_sector,'other') business_sector,
   get_json_object(amenity_json,'$.ext_attributes.primaryType') primary_type,
   get_json_object(amenity_json,'$.amenities') amenities_json,
   get_json_object(amenity_json,'$.ext_attributes') ext_attributes_json,
   types_json place_types_json,
   CAST(NULL AS TIMESTAMP) source_updated_at,crawled_at collected_at,
   CAST(NULL AS INT) source_priority,CAST(NULL AS DOUBLE) confidence_score,
   event_id _batch_id,current_timestamp() _now
 FROM {STAGE}
 WHERE run_id='{run_id}' AND rn_by_poi=1 AND amenity_json IS NOT NULL
) src
ON tgt.poi_id=src.poi_id AND tgt.source_name=src.source_name
WHEN MATCHED THEN UPDATE SET tgt.source_poi_id=src.source_poi_id,tgt.business_sector=src.business_sector,tgt.primary_type=src.primary_type,tgt.amenities_json=src.amenities_json,tgt.ext_attributes_json=src.ext_attributes_json,tgt.place_types_json=src.place_types_json,tgt.source_updated_at=src.source_updated_at,tgt.collected_at=src.collected_at,tgt.source_priority=src.source_priority,tgt.confidence_score=src.confidence_score,tgt._batch_id=src._batch_id,tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT(poi_id,source_name,source_poi_id,business_sector,primary_type,amenities_json,ext_attributes_json,place_types_json,source_updated_at,collected_at,source_priority,confidence_score,_batch_id,_created_at,_updated_at)
VALUES(src.poi_id,src.source_name,src.source_poi_id,src.business_sector,src.primary_type,src.amenities_json,src.ext_attributes_json,src.place_types_json,src.source_updated_at,src.collected_at,src.source_priority,src.confidence_score,src._batch_id,src._now,src._now)
""")

spark.sql(f"""
MERGE INTO poi_amenity_current tgt
USING (
 WITH impacted AS (SELECT DISTINCT poi_id FROM {STAGE} WHERE run_id='{run_id}'),
 base AS (
   SELECT a.*,ROW_NUMBER() OVER(PARTITION BY a.poi_id ORDER BY COALESCE(a.source_priority,0) DESC,a.collected_at DESC) rn
   FROM poi_amenity_source a JOIN impacted i ON i.poi_id=a.poi_id
 ), agg AS (
   SELECT poi_id,
     max(CASE WHEN rn=1 THEN business_sector END) business_sector,
     max(CASE WHEN rn=1 THEN primary_type END) primary_type,
     max(CASE WHEN rn=1 THEN amenities_json END) resolved_amenities_json,
     concat('{{',concat_ws(',',collect_list(concat('"',source_name,'":',COALESCE(amenities_json,'null')))),'}}') amenity_sources_json,
     max(CASE WHEN rn=1 THEN ext_attributes_json END) ext_attributes_json,
     max(CASE WHEN rn=1 THEN place_types_json END) place_types_json,
     max(CASE WHEN rn=1 THEN collected_at END) resolved_at,
     max(CASE WHEN rn=1 THEN _batch_id END) _batch_id
   FROM base GROUP BY poi_id
 ) SELECT *,CAST(NULL AS STRING) amenity_resolution_info_json,current_timestamp() _now FROM agg
) src
ON tgt.poi_id=src.poi_id
WHEN MATCHED THEN UPDATE SET tgt.business_sector=src.business_sector,tgt.primary_type=src.primary_type,tgt.resolved_amenities_json=src.resolved_amenities_json,tgt.amenity_sources_json=src.amenity_sources_json,tgt.amenity_resolution_info_json=src.amenity_resolution_info_json,tgt.ext_attributes_json=src.ext_attributes_json,tgt.place_types_json=src.place_types_json,tgt.resolved_at=src.resolved_at,tgt._batch_id=src._batch_id,tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT(poi_id,business_sector,primary_type,resolved_amenities_json,amenity_sources_json,amenity_resolution_info_json,ext_attributes_json,place_types_json,resolved_at,_batch_id,_created_at,_updated_at)
VALUES(src.poi_id,src.business_sector,src.primary_type,src.resolved_amenities_json,src.amenity_sources_json,src.amenity_resolution_info_json,src.ext_attributes_json,src.place_types_json,src.resolved_at,src._batch_id,src._now,src._now)
""")

notebookutils.notebook.exit("SUCCESS")


In [ ]:
# Fabric Notebook: NB_20_SILVER_POI_COMPONENTS
# READ/WRITE: Silver only
# Uses _stage_poi_language as the canonical incremental input.

from pyspark.sql import functions as F

# NB20 EXECUTION CONTRACT
# Default Lakehouse MUST be lh_vv_silver.
# All Silver source/target tables intentionally use single-part names.
# Do not change them back to lh_vv_silver.dbo.* in this notebook.
run_id = globals().get("run_id", "DEV_RUN")
STAGE = "_stage_poi_language"

# Namespace/schema preflight. spark.table(single_part) must resolve against the
# notebook's default Lakehouse (lh_vv_silver).
_required_silver_tables = [
    STAGE,
    "poi_address",
    "poi_contact",
    "poi_price",
    "poi_content",
    "poi_rating",
    "poi_review",
    "poi_media",
    "poi_opening_hours",
    "poi_policy",
    "poi_amenity_source",
    "poi_amenity_current",
]
for _tbl in _required_silver_tables:
    try:
        spark.table(_tbl).limit(0).collect()
    except Exception as _e:
        raise RuntimeError(
            f"NB20_SILVER_TABLE_RESOLUTION_FAILED: table={_tbl}. "
            "Attach/default this notebook to Lakehouse lh_vv_silver. "
            f"Original error: {_e}"
        ) from _e

print("NB20 Silver namespace preflight = OK (single-part tables, default Lakehouse lh_vv_silver)")

# Register any existing mapping UDFs from V1 here unchanged if they are still required:
#   fabric_map_sector(primaryType, types_json)
#   fabric_map_category(primaryType, types_json)
#   fabric_build_amenities_from_raw(sector, category, raw_data_json)
# This version keeps business mapping semantics unchanged; only the multilingual input stage changes.

# -----------------------------------------------------------------------------
# 1. poi_address - scalar, latest per poi_id
# -----------------------------------------------------------------------------
spark.sql(f"""
MERGE INTO poi_address tgt
USING (
  SELECT *, current_timestamp() _now FROM (
    SELECT
      poi_id,
      get_json_object(address_json,'$.address_line1') address_line1,
      get_json_object(address_json,'$.address_line2') address_line2,
      get_json_object(address_json,'$.ward') ward,
      get_json_object(address_json,'$.district') district,
      get_json_object(address_json,'$.city') city,
      get_json_object(address_json,'$.state_province') state_province,
      get_json_object(address_json,'$.country_code') country_code,
      get_json_object(address_json,'$.postal_code') postal_code,
      get_json_object(address_json,'$.short_address') short_address,
      get_json_object(address_json,'$.full_address') full_address,
      get_json_object(address_json,'$.address_viet_map') address_viet_map,
      CAST(get_json_object(address_json,'$.lat') AS DOUBLE) lat,
      CAST(get_json_object(address_json,'$.lng') AS DOUBLE) lng,
      get_json_object(address_json,'$.geohash') geohash,
      get_json_object(address_json,'$.h3_index_r8') h3_res8,
      get_json_object(address_json,'$.plus_code') plus_code,
      CAST(NULL AS DOUBLE) entrance_lat,
      CAST(NULL AS DOUBLE) entrance_lng,
      event_id _batch_id
    FROM {STAGE}
    WHERE run_id='{run_id}' AND rn_by_poi=1
  )
) src
ON tgt.poi_id=src.poi_id
WHEN MATCHED THEN UPDATE SET
  tgt.address_line1=src.address_line1, tgt.address_line2=src.address_line2,
  tgt.ward=src.ward, tgt.district=src.district, tgt.city=src.city,
  tgt.state_province=src.state_province, tgt.country_code=src.country_code,
  tgt.postal_code=src.postal_code, tgt.short_address=src.short_address,
  tgt.full_address=src.full_address, tgt.address_viet_map=src.address_viet_map,
  tgt.lat=src.lat, tgt.lng=src.lng, tgt.geohash=src.geohash, tgt.h3_res8=src.h3_res8,
  tgt.plus_code=src.plus_code, tgt.entrance_lat=src.entrance_lat, tgt.entrance_lng=src.entrance_lng,
  tgt._batch_id=src._batch_id, tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT (
  poi_id,address_line1,address_line2,ward,district,city,state_province,country_code,postal_code,
  short_address,full_address,address_viet_map,lat,lng,geohash,h3_res8,plus_code,entrance_lat,entrance_lng,
  _batch_id,_created_at,_updated_at
) VALUES (
  src.poi_id,src.address_line1,src.address_line2,src.ward,src.district,src.city,src.state_province,
  src.country_code,src.postal_code,src.short_address,src.full_address,src.address_viet_map,
  src.lat,src.lng,src.geohash,src.h3_res8,src.plus_code,src.entrance_lat,src.entrance_lng,
  src._batch_id,src._now,src._now
)
""")

# -----------------------------------------------------------------------------
# 2. poi_contact - preserve V1 semantics: no update if contact block absent.
# -----------------------------------------------------------------------------
spark.sql(f"""
MERGE INTO poi_contact tgt
USING (
 SELECT *,current_timestamp() _now FROM (
  SELECT
    poi_id,
    get_json_object(contact_json,'$.phone') phone,
    CASE WHEN get_json_object(contact_json,'$.hotline') IS NOT NULL
         THEN array(get_json_object(contact_json,'$.hotline'))
         ELSE CAST(NULL AS ARRAY<STRING>) END secondary_phones,
    get_json_object(contact_json,'$.email') email,
    get_json_object(contact_json,'$.website_url') website_url,
    COALESCE(
      get_json_object(contact_json,'$.booking_links.agoda'),
      get_json_object(contact_json,'$.booking_links.booking_com'),
      get_json_object(contact_json,'$.booking_links.traveloka'),
      get_json_object(contact_json,'$.booking_links.airbnb'),
      get_json_object(contact_json,'$.booking_links.vntrip')
    ) booking_url,
    CAST(NULL AS STRING) menu_url,
    get_json_object(contact_json,'$.social_links.facebook') facebook_url,
    get_json_object(contact_json,'$.social_links.instagram') instagram_url,
    get_json_object(contact_json,'$.social_links.tiktok') tiktok_url,
    CAST(NULL AS STRING) whatsapp_number,
    event_id _batch_id
  FROM {STAGE}
  WHERE run_id='{run_id}' AND rn_by_poi=1 AND contact_json IS NOT NULL
 )
) src
ON tgt.poi_id=src.poi_id
WHEN MATCHED THEN UPDATE SET
 tgt.phone=src.phone,tgt.secondary_phones=src.secondary_phones,tgt.email=src.email,
 tgt.website_url=src.website_url,tgt.booking_url=src.booking_url,tgt.menu_url=src.menu_url,
 tgt.facebook_url=src.facebook_url,tgt.instagram_url=src.instagram_url,tgt.tiktok_url=src.tiktok_url,
 tgt.whatsapp_number=src.whatsapp_number,tgt._batch_id=src._batch_id,tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT (
 poi_id,phone,secondary_phones,email,website_url,booking_url,menu_url,facebook_url,instagram_url,tiktok_url,
 whatsapp_number,_batch_id,_created_at,_updated_at
) VALUES (
 src.poi_id,src.phone,src.secondary_phones,src.email,src.website_url,src.booking_url,src.menu_url,
 src.facebook_url,src.instagram_url,src.tiktok_url,src.whatsapp_number,src._batch_id,src._now,src._now
)
""")

# -----------------------------------------------------------------------------
# 3. poi_price
# -----------------------------------------------------------------------------
spark.sql(f"""
MERGE INTO poi_price tgt
USING (
 SELECT *,current_timestamp() _now FROM (
  SELECT poi_id,
    COALESCE(get_json_object(price_json,'$.currency'),get_json_object(raw_data_json,'$.priceRange.startPrice.currencyCode')) currency,
    COALESCE(get_json_object(price_json,'$.price_level'),get_json_object(raw_data_json,'$.priceLevel')) price_level,
    COALESCE(CAST(get_json_object(price_json,'$.price_min') AS DOUBLE),CAST(get_json_object(raw_data_json,'$.priceRange.startPrice.units') AS DOUBLE)) price_min,
    COALESCE(CAST(get_json_object(price_json,'$.price_max') AS DOUBLE),CAST(get_json_object(raw_data_json,'$.priceRange.endPrice.units') AS DOUBLE)) price_max,
    event_id _batch_id
  FROM {STAGE}
  WHERE run_id='{run_id}' AND rn_by_poi=1 AND (price_json IS NOT NULL OR raw_data_json IS NOT NULL)
 )
) src
ON tgt.poi_id=src.poi_id
WHEN MATCHED THEN UPDATE SET tgt.currency=src.currency,tgt.price_level=src.price_level,tgt.price_min=src.price_min,tgt.price_max=src.price_max,tgt._batch_id=src._batch_id,tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT (poi_id,currency,price_level,price_min,price_max,_batch_id,_created_at,_updated_at)
VALUES(src.poi_id,src.currency,src.price_level,src.price_min,src.price_max,src._batch_id,src._now,src._now)
""")

# -----------------------------------------------------------------------------
# 4. poi_content - only content matching the logical POI language is selected first.
# If no matching locale exists, preserve V1 fallback behavior by keeping all locale rows.
# -----------------------------------------------------------------------------
spark.sql(f"""
MERGE INTO poi_content tgt
USING (
 WITH exploded AS (
   SELECT s.poi_id,s.lang,s.event_id,s.crawled_at,c.*
   FROM {STAGE} s
   LATERAL VIEW explode(from_json(s.content_json,
     'ARRAY<STRUCT<content_id:STRING,locale:STRING,seo_title:STRING,seo_description:STRING,text:STRING,service_bio:STRING,service_story:STRING,service_usps:ARRAY<STRING>,keywords:ARRAY<STRING>,local_tips:STRING,important_notice:STRING,is_original:BOOLEAN,translation_status:STRING,created_at:STRING>>')) x AS c
   WHERE s.run_id='{run_id}' AND s.content_json IS NOT NULL
 ), ranked AS (
   SELECT *, ROW_NUMBER() OVER(PARTITION BY poi_id,COALESCE(locale,lang) ORDER BY crawled_at DESC) rn
   FROM exploded
 )
 SELECT
   poi_id,COALESCE(locale,lang) locale,
   NULL AS _dummy,
   current_timestamp() _now,
   event_id _batch_id,
   seo_title display_name_override,
   text,seo_description,service_story,service_bio,service_usps,important_notice,keywords,created_at
 FROM ranked WHERE rn=1
) src
ON tgt.poi_id=src.poi_id AND tgt.locale=src.locale
WHEN MATCHED THEN UPDATE SET
 tgt.display_name=COALESCE(tgt.display_name,src.display_name_override),
 tgt.description_short=COALESCE(src.text,src.seo_description),
 tgt.description_long=src.service_story,tgt.service_bio=src.service_bio,tgt.service_usps=src.service_usps,
 tgt.important_notice=src.important_notice,tgt.tags_text=src.keywords,
 tgt.seo_title=src.display_name_override,tgt.seo_description=src.seo_description,
 tgt.content_updated_at=CAST(src.created_at AS TIMESTAMP),tgt._batch_id=src._batch_id,tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT(
 poi_id,locale,display_name,description_short,description_long,service_bio,service_usps,important_notice,tags_text,
 price_range_text,hours_weekday_text,seo_title,seo_description,content_updated_at,_batch_id,_created_at,_updated_at
) VALUES(
 src.poi_id,src.locale,src.display_name_override,COALESCE(src.text,src.seo_description),src.service_story,src.service_bio,
 src.service_usps,src.important_notice,src.keywords,NULL,NULL,src.display_name_override,src.seo_description,
 CAST(src.created_at AS TIMESTAMP),src._batch_id,src._now,src._now
)
""")

# -----------------------------------------------------------------------------
# 5. poi_rating
# -----------------------------------------------------------------------------
spark.sql(f"""
MERGE INTO poi_rating tgt
USING (
 SELECT *,current_timestamp() _now FROM (
  SELECT poi_id,
   CAST(get_json_object(rating_json,'$.rating_overall') AS DECIMAL(3,1)) overall,
   COALESCE(CAST(get_json_object(rating_json,'$.rating_count') AS BIGINT),0) total_reviews,
   get_json_object(rating_json,'$.rating_label') score_label,
   CAST(get_json_object(rating_json,'$.rating_weighted') AS DECIMAL(4,2)) rating_weighted,
   to_json(array(named_struct('source',get_json_object(rating_json,'$.source'),'rating_overall',CAST(get_json_object(rating_json,'$.rating_overall') AS DOUBLE),'rating_count',CAST(get_json_object(rating_json,'$.rating_count') AS BIGINT)))) sources_json,
   rating_json breakdown_json,
   CAST(NULL AS DECIMAL(4,3)) sentiment_avg,
   CAST(get_json_object(rating_json,'$.last_review_at') AS TIMESTAMP) last_review_at,
   event_id _batch_id
  FROM {STAGE} WHERE run_id='{run_id}' AND rn_by_poi=1 AND rating_json IS NOT NULL
 )
) src
ON tgt.poi_id=src.poi_id
WHEN MATCHED THEN UPDATE SET tgt.overall=src.overall,tgt.total_reviews=src.total_reviews,tgt.score_label=src.score_label,tgt.rating_weighted=src.rating_weighted,tgt.sources_json=src.sources_json,tgt.breakdown_json=src.breakdown_json,tgt.sentiment_avg=src.sentiment_avg,tgt.last_review_at=src.last_review_at,tgt._batch_id=src._batch_id,tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT(poi_id,overall,total_reviews,score_label,rating_weighted,sources_json,breakdown_json,sentiment_avg,last_review_at,_batch_id,_created_at,_updated_at)
VALUES(src.poi_id,src.overall,src.total_reviews,src.score_label,src.rating_weighted,src.sources_json,src.breakdown_json,src.sentiment_avg,src.last_review_at,src._batch_id,src._now,src._now)
""")

# -----------------------------------------------------------------------------
# 6. poi_review - IMPORTANT V2 change: review_id includes poi_id so shared root reviews
# can be replicated safely to N language-specific POIs.
# -----------------------------------------------------------------------------
spark.sql(f"""
MERGE INTO poi_review tgt
USING (
 SELECT *,current_timestamp() _now FROM (
  SELECT
   s.poi_id,s.source_name,
   sha2(concat(s.source_name,s.source_id,COALESCE(r.author_name,''),COALESCE(r.time,'')),256) source_review_id,
   sha2(concat(s.poi_id,'|',s.source_name,'|',s.source_id,'|',COALESCE(r.author_name,''),'|',COALESCE(r.time,'')),256) review_id,
   s.lang language,r.author_name reviewer_name,CAST(NULL AS STRING) reviewer_id_hash,
   CAST(r.rating AS DECIMAL(3,1)) rating_score,CAST(NULL AS STRING) review_title,r.text review_text,
   CAST(NULL AS ARRAY<STRING>) review_media_urls,CAST(r.time AS DATE) review_date,
   CAST(NULL AS STRING) travel_type,CAST(NULL AS BIGINT) helpful_count,
   CAST(NULL AS DOUBLE) fraud_score,CAST(NULL AS ARRAY<STRING>) fraud_reasons,true is_visible,
   CAST(NULL AS STRING) response_text,CAST(NULL AS TIMESTAMP) responded_at,CAST(r.time AS TIMESTAMP) published_at,
   s.event_id _batch_id,s.crawled_at,
   ROW_NUMBER() OVER(PARTITION BY s.poi_id,sha2(concat(s.source_name,s.source_id,COALESCE(r.author_name,''),COALESCE(r.time,'')),256) ORDER BY s.crawled_at DESC) rn
  FROM {STAGE} s
  LATERAL VIEW explode(from_json(s.review_json,'ARRAY<STRUCT<author_name:STRING,rating:INT,source:STRING,text:STRING,time:STRING>>')) x AS r
  WHERE s.run_id='{run_id}' AND s.review_json IS NOT NULL AND r.author_name IS NOT NULL AND r.time IS NOT NULL
 ) WHERE rn=1
) src
ON tgt.review_id=src.review_id
WHEN MATCHED THEN UPDATE SET tgt.poi_id=src.poi_id,tgt.is_visible=src.is_visible,tgt.language=src.language,tgt.helpful_count=src.helpful_count,tgt.fraud_score=src.fraud_score,tgt.fraud_reasons=src.fraud_reasons,tgt.response_text=src.response_text,tgt.responded_at=src.responded_at,tgt._batch_id=src._batch_id,tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT(poi_id,source_name,source_review_id,review_id,language,reviewer_name,reviewer_id_hash,rating_score,review_title,review_text,review_media_urls,review_date,travel_type,helpful_count,fraud_score,fraud_reasons,is_visible,response_text,responded_at,published_at,_batch_id,_created_at,_updated_at)
VALUES(src.poi_id,src.source_name,src.source_review_id,src.review_id,src.language,src.reviewer_name,src.reviewer_id_hash,src.rating_score,src.review_title,src.review_text,src.review_media_urls,src.review_date,src.travel_type,src.helpful_count,src.fraud_score,src.fraud_reasons,src.is_visible,src.response_text,src.responded_at,src.published_at,src._batch_id,src._now,src._now)
""")

# -----------------------------------------------------------------------------
# 7. poi_media - canonical cumulative media dedup (V5)
#
# AUTHORITATIVE BUSINESS IDENTITY
#   source media = source_name + source_id + TRIM(photo_api_uri)
#   Silver row   = poi_id + media_dedup_key
#
# IMPORTANT
# - media_id/blob_url are storage identities, NOT business identities.
# - Existing canonical media_id/blob_url are preserved across later crawls.
# - New photo_api_uri values append new media.
# - Historical physical duplicates are reconciled safely:
#     * same poi_id + same media_id -> keep one canonical row
#     * same poi_id + same non-empty media_dedup_key -> keep one canonical row
# - We do NOT invent a canonical photo_api_uri key for historical rows when the
#   source POI source_id cannot be proven. Such rows retain LEGACY_* keys until
#   a current crawl provides the canonical source-photo identity.
# -----------------------------------------------------------------------------

MEDIA_TABLE = "poi_media"

# Ensure V5 contract columns exist.
for _col_def in ["photo_api_uri STRING", "media_dedup_key STRING"]:
    try:
        spark.sql(f"ALTER TABLE {MEDIA_TABLE} ADD COLUMNS ({_col_def})")
    except Exception:
        pass

# --------------------------------------------------------------------------
# 7A. Current-run canonical media source
# --------------------------------------------------------------------------
# Parse once and materialize as a temp view so all subsequent reconciliation
# and MERGE operations use exactly the same identity implementation.
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v_nb20_media_current AS
WITH exploded AS (
    SELECT
        s.poi_id,
        s.source_name,
        s.source_id,
        s.event_id AS _batch_id,
        s.crawled_at,
        m.*,
        TRIM(m.photo_api_uri) AS normalized_photo_api_uri,
        SHA2(
            CONCAT_WS(
                '|',
                TRIM(s.source_name),
                TRIM(s.source_id),
                TRIM(m.photo_api_uri)
            ),
            256
        ) AS canonical_media_dedup_key
    FROM {STAGE} s
    LATERAL VIEW explode(from_json(
        s.media_json,
        'ARRAY<STRUCT<media_id:STRING,media_fingerprint:STRING,photo_api_uri:STRING,media_type:STRING,blob_url:STRING,original_url:STRING,thumbnail_url:STRING,processing_status:STRING,category:STRING,is_blessed:BOOLEAN,display_order:INT,photographer:STRING,license:STRING,source_field:STRING,source:STRING,width_px:INT,height_px:INT,created_at:STRING,updated_at:STRING>>'
    )) x AS m
    WHERE s.run_id = '{run_id}'
      AND s.media_json IS NOT NULL
      AND m.media_id IS NOT NULL
      AND m.photo_api_uri IS NOT NULL
      AND TRIM(m.photo_api_uri) <> ''
),
ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY
                poi_id,
                source_name,
                source_id,
                normalized_photo_api_uri
            ORDER BY
                crawled_at ASC,
                COALESCE(display_order, 999999) ASC,
                media_id ASC
        ) AS rn
    FROM exploded
)
SELECT *
FROM ranked
WHERE rn = 1
""")

# --------------------------------------------------------------------------
# 7B. Reconcile existing rows using current crawl evidence
# --------------------------------------------------------------------------
# If the exact same media_id already exists for this POI, the current crawl
# proves its canonical source identity. Update that historical row's key rather
# than inserting another physical row.
#
# This fixes the 20 -> 40 issue where one historical row had LEGACY_* identity
# and the current crawl generated the canonical photo_api_uri identity.
spark.sql(f"""
MERGE INTO {MEDIA_TABLE} tgt
USING (
    SELECT
        poi_id,
        media_id,
        source_name,
        source_id,
        normalized_photo_api_uri,
        canonical_media_dedup_key
    FROM v_nb20_media_current
) src
ON  tgt.poi_id = src.poi_id
AND tgt.media_id = src.media_id
WHEN MATCHED THEN UPDATE SET
    tgt.media_dedup_key = src.canonical_media_dedup_key,
    tgt.photo_api_uri = COALESCE(tgt.photo_api_uri, src.normalized_photo_api_uri),
    tgt.source_name = COALESCE(tgt.source_name, src.source_name),
    tgt.source_media_id = COALESCE(tgt.source_media_id, src.source_id),
    tgt._updated_at = current_timestamp()
""")

# --------------------------------------------------------------------------
# 7C. Historical physical duplicate cleanup
# --------------------------------------------------------------------------
# Delta DELETE with correlated subqueries is not Fabric-safe. Build the exact
# duplicate row set in temp views and use MERGE ... WHEN MATCHED THEN DELETE.
#
# First pass: same POI + same media_id. This is safe because media_id identifies
# the same stored media row and directly addresses already-materialized doubles.
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v_nb20_media_dup_media_id AS
WITH ranked AS (
    SELECT
        poi_id,
        media_id,
        media_dedup_key,
        blob_url,
        _created_at,
        _updated_at,
        ROW_NUMBER() OVER (
            PARTITION BY poi_id, media_id
            ORDER BY
                CASE WHEN media_dedup_key IS NOT NULL
                           AND media_dedup_key NOT LIKE 'LEGACY_%'
                     THEN 0 ELSE 1 END,
                _created_at ASC NULLS LAST,
                _updated_at ASC NULLS LAST,
                blob_url ASC NULLS LAST
        ) AS rn
    FROM {MEDIA_TABLE}
    WHERE media_id IS NOT NULL
)
SELECT
    poi_id,
    media_id,
    media_dedup_key,
    blob_url,
    _created_at,
    _updated_at
FROM ranked
WHERE rn > 1
""")

# Use a deterministic row signature for DELETE matching. COALESCE is required
# because Delta equality with NULL would otherwise fail to match.
spark.sql(f"""
MERGE INTO {MEDIA_TABLE} tgt
USING v_nb20_media_dup_media_id src
ON  tgt.poi_id = src.poi_id
AND tgt.media_id = src.media_id
AND COALESCE(tgt.media_dedup_key, '') = COALESCE(src.media_dedup_key, '')
AND COALESCE(tgt.blob_url, '') = COALESCE(src.blob_url, '')
AND COALESCE(CAST(tgt._created_at AS STRING), '') = COALESCE(CAST(src._created_at AS STRING), '')
AND COALESCE(CAST(tgt._updated_at AS STRING), '') = COALESCE(CAST(src._updated_at AS STRING), '')
WHEN MATCHED THEN DELETE
""")

# Second pass: same canonical key but regenerated media_id/blob_url across crawls.
# Keep the earliest canonical storage row; delete later physical duplicates.
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v_nb20_media_dup_canonical_key AS
WITH ranked AS (
    SELECT
        poi_id,
        media_id,
        media_dedup_key,
        blob_url,
        _created_at,
        _updated_at,
        ROW_NUMBER() OVER (
            PARTITION BY poi_id, media_dedup_key
            ORDER BY
                _created_at ASC NULLS LAST,
                _updated_at ASC NULLS LAST,
                media_id ASC
        ) AS rn
    FROM {MEDIA_TABLE}
    WHERE media_dedup_key IS NOT NULL
      AND TRIM(media_dedup_key) <> ''
      AND media_dedup_key NOT LIKE 'LEGACY_%'
)
SELECT
    poi_id,
    media_id,
    media_dedup_key,
    blob_url,
    _created_at,
    _updated_at
FROM ranked
WHERE rn > 1
""")

spark.sql(f"""
MERGE INTO {MEDIA_TABLE} tgt
USING v_nb20_media_dup_canonical_key src
ON  tgt.poi_id = src.poi_id
AND tgt.media_id = src.media_id
AND tgt.media_dedup_key = src.media_dedup_key
AND COALESCE(tgt.blob_url, '') = COALESCE(src.blob_url, '')
AND COALESCE(CAST(tgt._created_at AS STRING), '') = COALESCE(CAST(src._created_at AS STRING), '')
AND COALESCE(CAST(tgt._updated_at AS STRING), '') = COALESCE(CAST(src._updated_at AS STRING), '')
WHEN MATCHED THEN DELETE
""")

# --------------------------------------------------------------------------
# 7D. Canonical cumulative MERGE
# --------------------------------------------------------------------------
# At this point existing same-media_id rows have been upgraded to the canonical
# key and duplicate physical rows have been removed. The canonical MERGE now
# matches on poi_id + media_dedup_key only.
spark.sql(f"""
MERGE INTO {MEDIA_TABLE} tgt
USING (
    SELECT *, current_timestamp() AS _now
    FROM v_nb20_media_current
) src
ON  tgt.poi_id = src.poi_id
AND tgt.media_dedup_key = src.canonical_media_dedup_key

WHEN MATCHED THEN UPDATE SET
    -- Never replace canonical storage identity on a later crawl.
    tgt.photo_api_uri = COALESCE(tgt.photo_api_uri, src.normalized_photo_api_uri),
    tgt.processing_status = COALESCE(src.processing_status, tgt.processing_status),
    tgt.is_blessed = COALESCE(src.is_blessed, tgt.is_blessed, false),
    tgt.display_order = COALESCE(tgt.display_order, src.display_order),
    tgt.thumbnail_url = COALESCE(tgt.thumbnail_url, src.thumbnail_url),
    tgt.is_active = true,
    tgt.source_updated_at = GREATEST(
        tgt.source_updated_at,
        CAST(src.updated_at AS TIMESTAMP)
    ),
    tgt._batch_id = src._batch_id,
    tgt._updated_at = src._now

WHEN NOT MATCHED THEN INSERT(
    poi_id,
    media_id,
    media_dedup_key,
    photo_api_uri,
    source_name,
    source_media_id,
    media_type,
    blob_url,
    original_url,
    thumbnail_url,
    processing_status,
    category,
    is_blessed,
    display_order,
    photographer,
    license,
    source_field,
    width_px,
    height_px,
    caption,
    source_created_at,
    source_updated_at,
    is_active,
    _batch_id,
    _created_at,
    _updated_at
)
VALUES(
    src.poi_id,
    src.media_id,
    src.canonical_media_dedup_key,
    src.normalized_photo_api_uri,
    src.source_name,
    src.source_id,
    src.media_type,
    src.blob_url,
    src.original_url,
    src.thumbnail_url,
    src.processing_status,
    src.category,
    COALESCE(src.is_blessed, false),
    src.display_order,
    src.photographer,
    src.license,
    src.source_field,
    src.width_px,
    src.height_px,
    NULL,
    CAST(src.created_at AS TIMESTAMP),
    CAST(src.updated_at AS TIMESTAMP),
    true,
    src._batch_id,
    src._now,
    src._now
)
""")

# --------------------------------------------------------------------------
# 7E. Observability + invariant checks
# --------------------------------------------------------------------------
_media_metrics = spark.sql(f"""
WITH current_input AS (
    SELECT *
    FROM v_nb20_media_current
),
target_for_current_poi AS (
    SELECT m.*
    FROM {MEDIA_TABLE} m
    INNER JOIN (
        SELECT DISTINCT poi_id FROM current_input
    ) p ON p.poi_id = m.poi_id
),
dup_media_id AS (
    SELECT poi_id, media_id, COUNT(*) AS c
    FROM target_for_current_poi
    WHERE media_id IS NOT NULL
    GROUP BY poi_id, media_id
    HAVING COUNT(*) > 1
),
dup_key AS (
    SELECT poi_id, media_dedup_key, COUNT(*) AS c
    FROM target_for_current_poi
    WHERE media_dedup_key IS NOT NULL
      AND TRIM(media_dedup_key) <> ''
      AND media_dedup_key NOT LIKE 'LEGACY_%'
    GROUP BY poi_id, media_dedup_key
    HAVING COUNT(*) > 1
)
SELECT
    (SELECT COUNT(*) FROM current_input) AS media_current_unique_count,
    (SELECT COUNT(*) FROM target_for_current_poi) AS media_target_row_count,
    (SELECT COUNT(*) FROM dup_media_id) AS duplicate_media_id_groups,
    (SELECT COUNT(*) FROM dup_key) AS duplicate_canonical_key_groups
""").first()

print("NB20 media V5:", {
    "media_current_unique_count": _media_metrics["media_current_unique_count"],
    "media_target_row_count": _media_metrics["media_target_row_count"],
    "duplicate_media_id_groups": _media_metrics["duplicate_media_id_groups"],
    "duplicate_canonical_key_groups": _media_metrics["duplicate_canonical_key_groups"],
    "identity": "poi_id + SHA256(source_name|source_id|TRIM(photo_api_uri))",
})

if (_media_metrics["duplicate_media_id_groups"] or 0) > 0:
    raise RuntimeError(
        "NB20_MEDIA_INVARIANT_FAILED: duplicate poi_id + media_id remains after cleanup"
    )

if (_media_metrics["duplicate_canonical_key_groups"] or 0) > 0:
    raise RuntimeError(
        "NB20_MEDIA_INVARIANT_FAILED: duplicate poi_id + canonical media_dedup_key remains after cleanup"
    )

# -----------------------------------------------------------------------------
# 8. opening_hours - keep current V1 persisted semantics for this version.
# -----------------------------------------------------------------------------
spark.sql(f"""
MERGE INTO poi_opening_hours tgt
USING (
 SELECT poi_id,
   get_json_object(address_json,'$.timezone') timezone,
   COALESCE(CAST(get_json_object(opening_hours_json,'$.is_24_7') AS BOOLEAN),false) is_24_7,
   CAST(NULL AS BOOLEAN) open_now,
   get_json_object(opening_hours_json,'$.periods') periods_json,
   get_json_object(opening_hours_json,'$.holiday_exceptions') holiday_exceptions_json,
   CAST(get_json_object(opening_hours_json,'$.created_at') AS TIMESTAMP) source_last_changed_at,
   event_id _batch_id,current_timestamp() _now
 FROM {STAGE}
 WHERE run_id='{run_id}' AND rn_by_poi=1 AND opening_hours_json IS NOT NULL
) src
ON tgt.poi_id=src.poi_id
WHEN MATCHED THEN UPDATE SET tgt.timezone=src.timezone,tgt.is_24_7=src.is_24_7,tgt.open_now=src.open_now,tgt.periods_json=src.periods_json,tgt.holiday_exceptions_json=src.holiday_exceptions_json,tgt.source_last_changed_at=src.source_last_changed_at,tgt._batch_id=src._batch_id,tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT(poi_id,timezone,is_24_7,open_now,periods_json,holiday_exceptions_json,source_last_changed_at,_batch_id,_created_at,_updated_at)
VALUES(src.poi_id,src.timezone,src.is_24_7,src.open_now,src.periods_json,src.holiday_exceptions_json,src.source_last_changed_at,src._batch_id,src._now,src._now)
""")

# -----------------------------------------------------------------------------
# 9. poi_policy
# -----------------------------------------------------------------------------
spark.sql(f"""
MERGE INTO poi_policy tgt
USING (
 SELECT poi_id,policy_json policy_json,CAST(NULL AS TIMESTAMP) source_updated_at,event_id _batch_id,current_timestamp() _now
 FROM {STAGE} WHERE run_id='{run_id}' AND rn_by_poi=1 AND policy_json IS NOT NULL
) src ON tgt.poi_id=src.poi_id
WHEN MATCHED THEN UPDATE SET tgt.policy_json=src.policy_json,tgt.source_updated_at=src.source_updated_at,tgt._batch_id=src._batch_id,tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT(poi_id,policy_json,source_updated_at,_batch_id,_created_at,_updated_at)
VALUES(src.poi_id,src.policy_json,src.source_updated_at,src._batch_id,src._now,src._now)
""")

# -----------------------------------------------------------------------------
# 10/11. Amenity source/current.
# For V2 phase 1, raw amenity is replicated to each language-specific poi_id.
# Mapping expressions can be replaced by the existing V1 mapping UDF outputs.
# -----------------------------------------------------------------------------
spark.sql(f"""
MERGE INTO poi_amenity_source tgt
USING (
 SELECT poi_id,source_name,source_id source_poi_id,
   COALESCE(business_sector,'other') business_sector,
   get_json_object(amenity_json,'$.ext_attributes.primaryType') primary_type,
   get_json_object(amenity_json,'$.amenities') amenities_json,
   get_json_object(amenity_json,'$.ext_attributes') ext_attributes_json,
   types_json place_types_json,
   CAST(NULL AS TIMESTAMP) source_updated_at,crawled_at collected_at,
   CAST(NULL AS INT) source_priority,CAST(NULL AS DOUBLE) confidence_score,
   event_id _batch_id,current_timestamp() _now
 FROM {STAGE}
 WHERE run_id='{run_id}' AND rn_by_poi=1 AND amenity_json IS NOT NULL
) src
ON tgt.poi_id=src.poi_id AND tgt.source_name=src.source_name
WHEN MATCHED THEN UPDATE SET tgt.source_poi_id=src.source_poi_id,tgt.business_sector=src.business_sector,tgt.primary_type=src.primary_type,tgt.amenities_json=src.amenities_json,tgt.ext_attributes_json=src.ext_attributes_json,tgt.place_types_json=src.place_types_json,tgt.source_updated_at=src.source_updated_at,tgt.collected_at=src.collected_at,tgt.source_priority=src.source_priority,tgt.confidence_score=src.confidence_score,tgt._batch_id=src._batch_id,tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT(poi_id,source_name,source_poi_id,business_sector,primary_type,amenities_json,ext_attributes_json,place_types_json,source_updated_at,collected_at,source_priority,confidence_score,_batch_id,_created_at,_updated_at)
VALUES(src.poi_id,src.source_name,src.source_poi_id,src.business_sector,src.primary_type,src.amenities_json,src.ext_attributes_json,src.place_types_json,src.source_updated_at,src.collected_at,src.source_priority,src.confidence_score,src._batch_id,src._now,src._now)
""")

spark.sql(f"""
MERGE INTO poi_amenity_current tgt
USING (
 WITH impacted AS (SELECT DISTINCT poi_id FROM {STAGE} WHERE run_id='{run_id}'),
 base AS (
   SELECT a.*,ROW_NUMBER() OVER(PARTITION BY a.poi_id ORDER BY COALESCE(a.source_priority,0) DESC,a.collected_at DESC) rn
   FROM poi_amenity_source a JOIN impacted i ON i.poi_id=a.poi_id
 ), agg AS (
   SELECT poi_id,
     max(CASE WHEN rn=1 THEN business_sector END) business_sector,
     max(CASE WHEN rn=1 THEN primary_type END) primary_type,
     max(CASE WHEN rn=1 THEN amenities_json END) resolved_amenities_json,
     concat('{{',concat_ws(',',collect_list(concat('"',source_name,'":',COALESCE(amenities_json,'null')))),'}}') amenity_sources_json,
     max(CASE WHEN rn=1 THEN ext_attributes_json END) ext_attributes_json,
     max(CASE WHEN rn=1 THEN place_types_json END) place_types_json,
     max(CASE WHEN rn=1 THEN collected_at END) resolved_at,
     max(CASE WHEN rn=1 THEN _batch_id END) _batch_id
   FROM base GROUP BY poi_id
 ) SELECT *,CAST(NULL AS STRING) amenity_resolution_info_json,current_timestamp() _now FROM agg
) src
ON tgt.poi_id=src.poi_id
WHEN MATCHED THEN UPDATE SET tgt.business_sector=src.business_sector,tgt.primary_type=src.primary_type,tgt.resolved_amenities_json=src.resolved_amenities_json,tgt.amenity_sources_json=src.amenity_sources_json,tgt.amenity_resolution_info_json=src.amenity_resolution_info_json,tgt.ext_attributes_json=src.ext_attributes_json,tgt.place_types_json=src.place_types_json,tgt.resolved_at=src.resolved_at,tgt._batch_id=src._batch_id,tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT(poi_id,business_sector,primary_type,resolved_amenities_json,amenity_sources_json,amenity_resolution_info_json,ext_attributes_json,place_types_json,resolved_at,_batch_id,_created_at,_updated_at)
VALUES(src.poi_id,src.business_sector,src.primary_type,src.resolved_amenities_json,src.amenity_sources_json,src.amenity_resolution_info_json,src.ext_attributes_json,src.place_types_json,src.resolved_at,src._batch_id,src._now,src._now)
""")

notebookutils.notebook.exit("SUCCESS")


In [ ]:
# Fabric Notebook: NB_20_SILVER_POI_COMPONENTS
# READ/WRITE: Silver only
# Uses _stage_poi_language as the canonical incremental input.

from pyspark.sql import functions as F

run_id = globals().get("run_id", "DEV_RUN")
STAGE = "lh_vv_silver.dbo._stage_poi_language"

# Register any existing mapping UDFs from V1 here unchanged if they are still required:
#   fabric_map_sector(primaryType, types_json)
#   fabric_map_category(primaryType, types_json)
#   fabric_build_amenities_from_raw(sector, category, raw_data_json)
# This version keeps business mapping semantics unchanged; only the multilingual input stage changes.

# -----------------------------------------------------------------------------
# 1. poi_address - scalar, latest per poi_id
# -----------------------------------------------------------------------------
spark.sql(f"""
MERGE INTO lh_vv_silver.dbo.poi_address tgt
USING (
  SELECT *, current_timestamp() _now FROM (
    SELECT
      poi_id,
      get_json_object(address_json,'$.address_line1') address_line1,
      get_json_object(address_json,'$.address_line2') address_line2,
      get_json_object(address_json,'$.ward') ward,
      get_json_object(address_json,'$.district') district,
      get_json_object(address_json,'$.city') city,
      get_json_object(address_json,'$.state_province') state_province,
      get_json_object(address_json,'$.country_code') country_code,
      get_json_object(address_json,'$.postal_code') postal_code,
      get_json_object(address_json,'$.short_address') short_address,
      get_json_object(address_json,'$.full_address') full_address,
      get_json_object(address_json,'$.address_viet_map') address_viet_map,
      CAST(get_json_object(address_json,'$.lat') AS DOUBLE) lat,
      CAST(get_json_object(address_json,'$.lng') AS DOUBLE) lng,
      get_json_object(address_json,'$.geohash') geohash,
      get_json_object(address_json,'$.h3_index_r8') h3_res8,
      get_json_object(address_json,'$.plus_code') plus_code,
      CAST(NULL AS DOUBLE) entrance_lat,
      CAST(NULL AS DOUBLE) entrance_lng,
      event_id _batch_id
    FROM {STAGE}
    WHERE run_id='{run_id}' AND rn_by_poi=1
  )
) src
ON tgt.poi_id=src.poi_id
WHEN MATCHED THEN UPDATE SET
  tgt.address_line1=src.address_line1, tgt.address_line2=src.address_line2,
  tgt.ward=src.ward, tgt.district=src.district, tgt.city=src.city,
  tgt.state_province=src.state_province, tgt.country_code=src.country_code,
  tgt.postal_code=src.postal_code, tgt.short_address=src.short_address,
  tgt.full_address=src.full_address, tgt.address_viet_map=src.address_viet_map,
  tgt.lat=src.lat, tgt.lng=src.lng, tgt.geohash=src.geohash, tgt.h3_res8=src.h3_res8,
  tgt.plus_code=src.plus_code, tgt.entrance_lat=src.entrance_lat, tgt.entrance_lng=src.entrance_lng,
  tgt._batch_id=src._batch_id, tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT (
  poi_id,address_line1,address_line2,ward,district,city,state_province,country_code,postal_code,
  short_address,full_address,address_viet_map,lat,lng,geohash,h3_res8,plus_code,entrance_lat,entrance_lng,
  _batch_id,_created_at,_updated_at
) VALUES (
  src.poi_id,src.address_line1,src.address_line2,src.ward,src.district,src.city,src.state_province,
  src.country_code,src.postal_code,src.short_address,src.full_address,src.address_viet_map,
  src.lat,src.lng,src.geohash,src.h3_res8,src.plus_code,src.entrance_lat,src.entrance_lng,
  src._batch_id,src._now,src._now
)
""")

# -----------------------------------------------------------------------------
# 2. poi_contact - preserve V1 semantics: no update if contact block absent.
# -----------------------------------------------------------------------------
spark.sql(f"""
MERGE INTO lh_vv_silver.dbo.poi_contact tgt
USING (
 SELECT *,current_timestamp() _now FROM (
  SELECT
    poi_id,
    get_json_object(contact_json,'$.phone') phone,
    CASE WHEN get_json_object(contact_json,'$.hotline') IS NOT NULL
         THEN array(get_json_object(contact_json,'$.hotline'))
         ELSE CAST(NULL AS ARRAY<STRING>) END secondary_phones,
    get_json_object(contact_json,'$.email') email,
    get_json_object(contact_json,'$.website_url') website_url,
    COALESCE(
      get_json_object(contact_json,'$.booking_links.agoda'),
      get_json_object(contact_json,'$.booking_links.booking_com'),
      get_json_object(contact_json,'$.booking_links.traveloka'),
      get_json_object(contact_json,'$.booking_links.airbnb'),
      get_json_object(contact_json,'$.booking_links.vntrip')
    ) booking_url,
    CAST(NULL AS STRING) menu_url,
    get_json_object(contact_json,'$.social_links.facebook') facebook_url,
    get_json_object(contact_json,'$.social_links.instagram') instagram_url,
    get_json_object(contact_json,'$.social_links.tiktok') tiktok_url,
    CAST(NULL AS STRING) whatsapp_number,
    event_id _batch_id
  FROM {STAGE}
  WHERE run_id='{run_id}' AND rn_by_poi=1 AND contact_json IS NOT NULL
 )
) src
ON tgt.poi_id=src.poi_id
WHEN MATCHED THEN UPDATE SET
 tgt.phone=src.phone,tgt.secondary_phones=src.secondary_phones,tgt.email=src.email,
 tgt.website_url=src.website_url,tgt.booking_url=src.booking_url,tgt.menu_url=src.menu_url,
 tgt.facebook_url=src.facebook_url,tgt.instagram_url=src.instagram_url,tgt.tiktok_url=src.tiktok_url,
 tgt.whatsapp_number=src.whatsapp_number,tgt._batch_id=src._batch_id,tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT (
 poi_id,phone,secondary_phones,email,website_url,booking_url,menu_url,facebook_url,instagram_url,tiktok_url,
 whatsapp_number,_batch_id,_created_at,_updated_at
) VALUES (
 src.poi_id,src.phone,src.secondary_phones,src.email,src.website_url,src.booking_url,src.menu_url,
 src.facebook_url,src.instagram_url,src.tiktok_url,src.whatsapp_number,src._batch_id,src._now,src._now
)
""")

# -----------------------------------------------------------------------------
# 3. poi_price
# -----------------------------------------------------------------------------
spark.sql(f"""
MERGE INTO lh_vv_silver.dbo.poi_price tgt
USING (
 SELECT *,current_timestamp() _now FROM (
  SELECT poi_id,
    COALESCE(get_json_object(price_json,'$.currency'),get_json_object(raw_data_json,'$.priceRange.startPrice.currencyCode')) currency,
    COALESCE(get_json_object(price_json,'$.price_level'),get_json_object(raw_data_json,'$.priceLevel')) price_level,
    COALESCE(CAST(get_json_object(price_json,'$.price_min') AS DOUBLE),CAST(get_json_object(raw_data_json,'$.priceRange.startPrice.units') AS DOUBLE)) price_min,
    COALESCE(CAST(get_json_object(price_json,'$.price_max') AS DOUBLE),CAST(get_json_object(raw_data_json,'$.priceRange.endPrice.units') AS DOUBLE)) price_max,
    event_id _batch_id
  FROM {STAGE}
  WHERE run_id='{run_id}' AND rn_by_poi=1 AND (price_json IS NOT NULL OR raw_data_json IS NOT NULL)
 )
) src
ON tgt.poi_id=src.poi_id
WHEN MATCHED THEN UPDATE SET tgt.currency=src.currency,tgt.price_level=src.price_level,tgt.price_min=src.price_min,tgt.price_max=src.price_max,tgt._batch_id=src._batch_id,tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT (poi_id,currency,price_level,price_min,price_max,_batch_id,_created_at,_updated_at)
VALUES(src.poi_id,src.currency,src.price_level,src.price_min,src.price_max,src._batch_id,src._now,src._now)
""")

# -----------------------------------------------------------------------------
# 4. poi_content - only content matching the logical POI language is selected first.
# If no matching locale exists, preserve V1 fallback behavior by keeping all locale rows.
# -----------------------------------------------------------------------------
spark.sql(f"""
MERGE INTO lh_vv_silver.dbo.poi_content tgt
USING (
 WITH exploded AS (
   SELECT s.poi_id,s.lang,s.event_id,s.crawled_at,c.*
   FROM {STAGE} s
   LATERAL VIEW explode(from_json(s.content_json,
     'ARRAY<STRUCT<content_id:STRING,locale:STRING,seo_title:STRING,seo_description:STRING,text:STRING,service_bio:STRING,service_story:STRING,service_usps:ARRAY<STRING>,keywords:ARRAY<STRING>,local_tips:STRING,important_notice:STRING,is_original:BOOLEAN,translation_status:STRING,created_at:STRING>>')) x AS c
   WHERE s.run_id='{run_id}' AND s.content_json IS NOT NULL
 ), ranked AS (
   SELECT *, ROW_NUMBER() OVER(PARTITION BY poi_id,COALESCE(locale,lang) ORDER BY crawled_at DESC) rn
   FROM exploded
 )
 SELECT
   poi_id,COALESCE(locale,lang) locale,
   NULL AS _dummy,
   current_timestamp() _now,
   event_id _batch_id,
   seo_title display_name_override,
   text,seo_description,service_story,service_bio,service_usps,important_notice,keywords,created_at
 FROM ranked WHERE rn=1
) src
ON tgt.poi_id=src.poi_id AND tgt.locale=src.locale
WHEN MATCHED THEN UPDATE SET
 tgt.display_name=COALESCE(tgt.display_name,src.display_name_override),
 tgt.description_short=COALESCE(src.text,src.seo_description),
 tgt.description_long=src.service_story,tgt.service_bio=src.service_bio,tgt.service_usps=src.service_usps,
 tgt.important_notice=src.important_notice,tgt.tags_text=src.keywords,
 tgt.seo_title=src.display_name_override,tgt.seo_description=src.seo_description,
 tgt.content_updated_at=CAST(src.created_at AS TIMESTAMP),tgt._batch_id=src._batch_id,tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT(
 poi_id,locale,display_name,description_short,description_long,service_bio,service_usps,important_notice,tags_text,
 price_range_text,hours_weekday_text,seo_title,seo_description,content_updated_at,_batch_id,_created_at,_updated_at
) VALUES(
 src.poi_id,src.locale,src.display_name_override,COALESCE(src.text,src.seo_description),src.service_story,src.service_bio,
 src.service_usps,src.important_notice,src.keywords,NULL,NULL,src.display_name_override,src.seo_description,
 CAST(src.created_at AS TIMESTAMP),src._batch_id,src._now,src._now
)
""")

# -----------------------------------------------------------------------------
# 5. poi_rating
# -----------------------------------------------------------------------------
spark.sql(f"""
MERGE INTO lh_vv_silver.dbo.poi_rating tgt
USING (
 SELECT *,current_timestamp() _now FROM (
  SELECT poi_id,
   CAST(get_json_object(rating_json,'$.rating_overall') AS DECIMAL(3,1)) overall,
   COALESCE(CAST(get_json_object(rating_json,'$.rating_count') AS BIGINT),0) total_reviews,
   get_json_object(rating_json,'$.rating_label') score_label,
   CAST(get_json_object(rating_json,'$.rating_weighted') AS DECIMAL(4,2)) rating_weighted,
   to_json(array(named_struct('source',get_json_object(rating_json,'$.source'),'rating_overall',CAST(get_json_object(rating_json,'$.rating_overall') AS DOUBLE),'rating_count',CAST(get_json_object(rating_json,'$.rating_count') AS BIGINT)))) sources_json,
   rating_json breakdown_json,
   CAST(NULL AS DECIMAL(4,3)) sentiment_avg,
   CAST(get_json_object(rating_json,'$.last_review_at') AS TIMESTAMP) last_review_at,
   event_id _batch_id
  FROM {STAGE} WHERE run_id='{run_id}' AND rn_by_poi=1 AND rating_json IS NOT NULL
 )
) src
ON tgt.poi_id=src.poi_id
WHEN MATCHED THEN UPDATE SET tgt.overall=src.overall,tgt.total_reviews=src.total_reviews,tgt.score_label=src.score_label,tgt.rating_weighted=src.rating_weighted,tgt.sources_json=src.sources_json,tgt.breakdown_json=src.breakdown_json,tgt.sentiment_avg=src.sentiment_avg,tgt.last_review_at=src.last_review_at,tgt._batch_id=src._batch_id,tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT(poi_id,overall,total_reviews,score_label,rating_weighted,sources_json,breakdown_json,sentiment_avg,last_review_at,_batch_id,_created_at,_updated_at)
VALUES(src.poi_id,src.overall,src.total_reviews,src.score_label,src.rating_weighted,src.sources_json,src.breakdown_json,src.sentiment_avg,src.last_review_at,src._batch_id,src._now,src._now)
""")

# -----------------------------------------------------------------------------
# 6. poi_review - IMPORTANT V2 change: review_id includes poi_id so shared root reviews
# can be replicated safely to N language-specific POIs.
# -----------------------------------------------------------------------------
spark.sql(f"""
MERGE INTO lh_vv_silver.dbo.poi_review tgt
USING (
 SELECT *,current_timestamp() _now FROM (
  SELECT
   s.poi_id,s.source_name,
   sha2(concat(s.source_name,s.source_id,COALESCE(r.author_name,''),COALESCE(r.time,'')),256) source_review_id,
   sha2(concat(s.poi_id,'|',s.source_name,'|',s.source_id,'|',COALESCE(r.author_name,''),'|',COALESCE(r.time,'')),256) review_id,
   s.lang language,r.author_name reviewer_name,CAST(NULL AS STRING) reviewer_id_hash,
   CAST(r.rating AS DECIMAL(3,1)) rating_score,CAST(NULL AS STRING) review_title,r.text review_text,
   CAST(NULL AS ARRAY<STRING>) review_media_urls,CAST(r.time AS DATE) review_date,
   CAST(NULL AS STRING) travel_type,CAST(NULL AS BIGINT) helpful_count,
   CAST(NULL AS DOUBLE) fraud_score,CAST(NULL AS ARRAY<STRING>) fraud_reasons,true is_visible,
   CAST(NULL AS STRING) response_text,CAST(NULL AS TIMESTAMP) responded_at,CAST(r.time AS TIMESTAMP) published_at,
   s.event_id _batch_id,s.crawled_at,
   ROW_NUMBER() OVER(PARTITION BY s.poi_id,sha2(concat(s.source_name,s.source_id,COALESCE(r.author_name,''),COALESCE(r.time,'')),256) ORDER BY s.crawled_at DESC) rn
  FROM {STAGE} s
  LATERAL VIEW explode(from_json(s.review_json,'ARRAY<STRUCT<author_name:STRING,rating:INT,source:STRING,text:STRING,time:STRING>>')) x AS r
  WHERE s.run_id='{run_id}' AND s.review_json IS NOT NULL AND r.author_name IS NOT NULL AND r.time IS NOT NULL
 ) WHERE rn=1
) src
ON tgt.review_id=src.review_id
WHEN MATCHED THEN UPDATE SET tgt.poi_id=src.poi_id,tgt.is_visible=src.is_visible,tgt.language=src.language,tgt.helpful_count=src.helpful_count,tgt.fraud_score=src.fraud_score,tgt.fraud_reasons=src.fraud_reasons,tgt.response_text=src.response_text,tgt.responded_at=src.responded_at,tgt._batch_id=src._batch_id,tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT(poi_id,source_name,source_review_id,review_id,language,reviewer_name,reviewer_id_hash,rating_score,review_title,review_text,review_media_urls,review_date,travel_type,helpful_count,fraud_score,fraud_reasons,is_visible,response_text,responded_at,published_at,_batch_id,_created_at,_updated_at)
VALUES(src.poi_id,src.source_name,src.source_review_id,src.review_id,src.language,src.reviewer_name,src.reviewer_id_hash,src.rating_score,src.review_title,src.review_text,src.review_media_urls,src.review_date,src.travel_type,src.helpful_count,src.fraud_score,src.fraud_reasons,src.is_visible,src.response_text,src.responded_at,src.published_at,src._batch_id,src._now,src._now)
""")

# -----------------------------------------------------------------------------
# 7. poi_media - cumulative media with cross-crawl source-photo dedup.
#
# Canonical identity:
#   source media = source_name + source_id + TRIM(photo_api_uri)
#   Silver row   = poi_id + media_dedup_key
#
# Existing canonical media_id/blob_url/original_url are preserved on MATCHED.
# New photo_api_uri values are appended; historical media are never deactivated.
# -----------------------------------------------------------------------------
for _col_def in ["photo_api_uri STRING", "media_dedup_key STRING"]:
    try:
        spark.sql(f"ALTER TABLE lh_vv_silver.dbo.poi_media ADD COLUMNS ({_col_def})")
    except Exception:
        pass

# Legacy rows only: preserve a stable fallback identity until historical data can
# be reconstructed from raw history. New ingestion below REQUIRES photo_api_uri.
spark.sql("""
UPDATE lh_vv_silver.dbo.poi_media
SET media_dedup_key = CASE
    WHEN photo_api_uri IS NOT NULL AND TRIM(photo_api_uri) <> ''
      THEN SHA2(CONCAT_WS('|',
             TRIM(COALESCE(source_name,'')),
             TRIM(COALESCE(source_media_id,'')),
             TRIM(photo_api_uri)), 256)
    WHEN original_url IS NOT NULL AND TRIM(original_url) <> ''
      THEN CONCAT('LEGACY_URL:', TRIM(original_url))
    ELSE CONCAT('LEGACY_ID:', media_id)
END
WHERE media_dedup_key IS NULL OR TRIM(media_dedup_key) = ''
""")

spark.sql(f"""
MERGE INTO lh_vv_silver.dbo.poi_media tgt
USING (
  WITH exploded AS (
    SELECT
      s.poi_id,
      s.source_name,
      s.source_id,
      s.event_id AS _batch_id,
      s.crawled_at,
      m.*,
      TRIM(m.photo_api_uri) AS normalized_photo_api_uri,
      SHA2(CONCAT_WS('|',
          TRIM(s.source_name),
          TRIM(s.source_id),
          TRIM(m.photo_api_uri)
      ), 256) AS media_dedup_key
    FROM {STAGE} s
    LATERAL VIEW explode(from_json(
      s.media_json,
      'ARRAY<STRUCT<media_id:STRING,media_fingerprint:STRING,photo_api_uri:STRING,media_type:STRING,blob_url:STRING,original_url:STRING,thumbnail_url:STRING,processing_status:STRING,category:STRING,is_blessed:BOOLEAN,display_order:INT,photographer:STRING,license:STRING,source_field:STRING,source:STRING,width_px:INT,height_px:INT,created_at:STRING,updated_at:STRING>>'
    )) x AS m
    WHERE s.run_id='{run_id}'
      AND s.media_json IS NOT NULL
      AND m.media_id IS NOT NULL
      AND m.photo_api_uri IS NOT NULL
      AND TRIM(m.photo_api_uri) <> ''
  ),
  ranked AS (
    SELECT *,
      ROW_NUMBER() OVER (
        PARTITION BY poi_id, source_name, source_id, normalized_photo_api_uri
        ORDER BY crawled_at ASC, COALESCE(display_order,999999) ASC, media_id ASC
      ) AS rn
    FROM exploded
  )
  SELECT *, current_timestamp() AS _now
  FROM ranked WHERE rn=1
) src
ON tgt.poi_id = src.poi_id
AND tgt.media_dedup_key = src.media_dedup_key
WHEN MATCHED THEN UPDATE SET
  tgt.photo_api_uri = COALESCE(tgt.photo_api_uri,src.normalized_photo_api_uri),
  tgt.processing_status = COALESCE(src.processing_status,tgt.processing_status),
  tgt.is_blessed = COALESCE(src.is_blessed,tgt.is_blessed,false),
  tgt.display_order = COALESCE(tgt.display_order,src.display_order),
  tgt.thumbnail_url = COALESCE(tgt.thumbnail_url,src.thumbnail_url),
  tgt.is_active = true,
  tgt.source_updated_at = GREATEST(tgt.source_updated_at,CAST(src.updated_at AS TIMESTAMP)),
  tgt._batch_id = src._batch_id,
  tgt._updated_at = src._now
WHEN NOT MATCHED THEN INSERT(
  poi_id,media_id,media_dedup_key,photo_api_uri,source_name,source_media_id,
  media_type,blob_url,original_url,thumbnail_url,processing_status,category,is_blessed,
  display_order,photographer,license,source_field,width_px,height_px,caption,
  source_created_at,source_updated_at,is_active,_batch_id,_created_at,_updated_at
)
VALUES(
  src.poi_id,src.media_id,src.media_dedup_key,src.normalized_photo_api_uri,
  src.source_name,src.source_id,
  src.media_type,src.blob_url,src.original_url,src.thumbnail_url,src.processing_status,
  src.category,COALESCE(src.is_blessed,false),src.display_order,src.photographer,src.license,
  src.source_field,src.width_px,src.height_px,NULL,CAST(src.created_at AS TIMESTAMP),
  CAST(src.updated_at AS TIMESTAMP),true,src._batch_id,src._now,src._now
)
""")

_media_metrics = spark.sql(f"""
WITH exploded AS (
  SELECT s.poi_id,s.source_name,s.source_id,TRIM(m.photo_api_uri) photo_api_uri
  FROM {STAGE} s
  LATERAL VIEW explode(from_json(
    s.media_json,
    'ARRAY<STRUCT<media_id:STRING,media_fingerprint:STRING,photo_api_uri:STRING,media_type:STRING,blob_url:STRING,original_url:STRING,thumbnail_url:STRING,processing_status:STRING,category:STRING,is_blessed:BOOLEAN,display_order:INT,photographer:STRING,license:STRING,source_field:STRING,source:STRING,width_px:INT,height_px:INT,created_at:STRING,updated_at:STRING>>'
  )) x AS m
  WHERE s.run_id='{run_id}' AND s.media_json IS NOT NULL AND m.media_id IS NOT NULL
)
SELECT COUNT(*) media_input_count,
       SUM(CASE WHEN photo_api_uri IS NULL OR photo_api_uri='' THEN 1 ELSE 0 END) media_missing_photo_api_uri_count,
       COUNT(DISTINCT CASE WHEN photo_api_uri IS NOT NULL AND photo_api_uri<>''
         THEN CONCAT_WS('|',poi_id,source_name,source_id,photo_api_uri) END) media_unique_source_photo_count
FROM exploded
""").first()

print("NB20 media dedup:", {
    "media_input_count": _media_metrics["media_input_count"],
    "media_missing_photo_api_uri_count": _media_metrics["media_missing_photo_api_uri_count"],
    "media_unique_source_photo_count": _media_metrics["media_unique_source_photo_count"],
    "identity": "poi_id + source_name + source_id + TRIM(photo_api_uri)"
})

if (_media_metrics["media_missing_photo_api_uri_count"] or 0) > 0:
    print("NB20 WARNING: media without photo_api_uri were skipped; verify Data Collector contract.")

# -----------------------------------------------------------------------------
# 8. opening_hours - keep current V1 persisted semantics for this version.
# -----------------------------------------------------------------------------
spark.sql(f"""
MERGE INTO lh_vv_silver.dbo.poi_opening_hours tgt
USING (
 SELECT poi_id,
   get_json_object(address_json,'$.timezone') timezone,
   COALESCE(CAST(get_json_object(opening_hours_json,'$.is_24_7') AS BOOLEAN),false) is_24_7,
   CAST(NULL AS BOOLEAN) open_now,
   get_json_object(opening_hours_json,'$.periods') periods_json,
   get_json_object(opening_hours_json,'$.holiday_exceptions') holiday_exceptions_json,
   CAST(get_json_object(opening_hours_json,'$.created_at') AS TIMESTAMP) source_last_changed_at,
   event_id _batch_id,current_timestamp() _now
 FROM {STAGE}
 WHERE run_id='{run_id}' AND rn_by_poi=1 AND opening_hours_json IS NOT NULL
) src
ON tgt.poi_id=src.poi_id
WHEN MATCHED THEN UPDATE SET tgt.timezone=src.timezone,tgt.is_24_7=src.is_24_7,tgt.open_now=src.open_now,tgt.periods_json=src.periods_json,tgt.holiday_exceptions_json=src.holiday_exceptions_json,tgt.source_last_changed_at=src.source_last_changed_at,tgt._batch_id=src._batch_id,tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT(poi_id,timezone,is_24_7,open_now,periods_json,holiday_exceptions_json,source_last_changed_at,_batch_id,_created_at,_updated_at)
VALUES(src.poi_id,src.timezone,src.is_24_7,src.open_now,src.periods_json,src.holiday_exceptions_json,src.source_last_changed_at,src._batch_id,src._now,src._now)
""")

# -----------------------------------------------------------------------------
# 9. poi_policy
# -----------------------------------------------------------------------------
spark.sql(f"""
MERGE INTO lh_vv_silver.dbo.poi_policy tgt
USING (
 SELECT poi_id,policy_json policy_json,CAST(NULL AS TIMESTAMP) source_updated_at,event_id _batch_id,current_timestamp() _now
 FROM {STAGE} WHERE run_id='{run_id}' AND rn_by_poi=1 AND policy_json IS NOT NULL
) src ON tgt.poi_id=src.poi_id
WHEN MATCHED THEN UPDATE SET tgt.policy_json=src.policy_json,tgt.source_updated_at=src.source_updated_at,tgt._batch_id=src._batch_id,tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT(poi_id,policy_json,source_updated_at,_batch_id,_created_at,_updated_at)
VALUES(src.poi_id,src.policy_json,src.source_updated_at,src._batch_id,src._now,src._now)
""")

# -----------------------------------------------------------------------------
# 10/11. Amenity source/current.
# For V2 phase 1, raw amenity is replicated to each language-specific poi_id.
# Mapping expressions can be replaced by the existing V1 mapping UDF outputs.
# -----------------------------------------------------------------------------
spark.sql(f"""
MERGE INTO lh_vv_silver.dbo.poi_amenity_source tgt
USING (
 SELECT poi_id,source_name,source_id source_poi_id,
   COALESCE(business_sector,'other') business_sector,
   get_json_object(amenity_json,'$.ext_attributes.primaryType') primary_type,
   get_json_object(amenity_json,'$.amenities') amenities_json,
   get_json_object(amenity_json,'$.ext_attributes') ext_attributes_json,
   types_json place_types_json,
   CAST(NULL AS TIMESTAMP) source_updated_at,crawled_at collected_at,
   CAST(NULL AS INT) source_priority,CAST(NULL AS DOUBLE) confidence_score,
   event_id _batch_id,current_timestamp() _now
 FROM {STAGE}
 WHERE run_id='{run_id}' AND rn_by_poi=1 AND amenity_json IS NOT NULL
) src
ON tgt.poi_id=src.poi_id AND tgt.source_name=src.source_name
WHEN MATCHED THEN UPDATE SET tgt.source_poi_id=src.source_poi_id,tgt.business_sector=src.business_sector,tgt.primary_type=src.primary_type,tgt.amenities_json=src.amenities_json,tgt.ext_attributes_json=src.ext_attributes_json,tgt.place_types_json=src.place_types_json,tgt.source_updated_at=src.source_updated_at,tgt.collected_at=src.collected_at,tgt.source_priority=src.source_priority,tgt.confidence_score=src.confidence_score,tgt._batch_id=src._batch_id,tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT(poi_id,source_name,source_poi_id,business_sector,primary_type,amenities_json,ext_attributes_json,place_types_json,source_updated_at,collected_at,source_priority,confidence_score,_batch_id,_created_at,_updated_at)
VALUES(src.poi_id,src.source_name,src.source_poi_id,src.business_sector,src.primary_type,src.amenities_json,src.ext_attributes_json,src.place_types_json,src.source_updated_at,src.collected_at,src.source_priority,src.confidence_score,src._batch_id,src._now,src._now)
""")

spark.sql(f"""
MERGE INTO lh_vv_silver.dbo.poi_amenity_current tgt
USING (
 WITH impacted AS (SELECT DISTINCT poi_id FROM {STAGE} WHERE run_id='{run_id}'),
 base AS (
   SELECT a.*,ROW_NUMBER() OVER(PARTITION BY a.poi_id ORDER BY COALESCE(a.source_priority,0) DESC,a.collected_at DESC) rn
   FROM lh_vv_silver.dbo.poi_amenity_source a JOIN impacted i ON i.poi_id=a.poi_id
 ), agg AS (
   SELECT poi_id,
     max(CASE WHEN rn=1 THEN business_sector END) business_sector,
     max(CASE WHEN rn=1 THEN primary_type END) primary_type,
     max(CASE WHEN rn=1 THEN amenities_json END) resolved_amenities_json,
     concat('{{',concat_ws(',',collect_list(concat('"',source_name,'":',COALESCE(amenities_json,'null')))),'}}') amenity_sources_json,
     max(CASE WHEN rn=1 THEN ext_attributes_json END) ext_attributes_json,
     max(CASE WHEN rn=1 THEN place_types_json END) place_types_json,
     max(CASE WHEN rn=1 THEN collected_at END) resolved_at,
     max(CASE WHEN rn=1 THEN _batch_id END) _batch_id
   FROM base GROUP BY poi_id
 ) SELECT *,CAST(NULL AS STRING) amenity_resolution_info_json,current_timestamp() _now FROM agg
) src
ON tgt.poi_id=src.poi_id
WHEN MATCHED THEN UPDATE SET tgt.business_sector=src.business_sector,tgt.primary_type=src.primary_type,tgt.resolved_amenities_json=src.resolved_amenities_json,tgt.amenity_sources_json=src.amenity_sources_json,tgt.amenity_resolution_info_json=src.amenity_resolution_info_json,tgt.ext_attributes_json=src.ext_attributes_json,tgt.place_types_json=src.place_types_json,tgt.resolved_at=src.resolved_at,tgt._batch_id=src._batch_id,tgt._updated_at=src._now
WHEN NOT MATCHED THEN INSERT(poi_id,business_sector,primary_type,resolved_amenities_json,amenity_sources_json,amenity_resolution_info_json,ext_attributes_json,place_types_json,resolved_at,_batch_id,_created_at,_updated_at)
VALUES(src.poi_id,src.business_sector,src.primary_type,src.resolved_amenities_json,src.amenity_sources_json,src.amenity_resolution_info_json,src.ext_attributes_json,src.place_types_json,src.resolved_at,src._batch_id,src._now,src._now)
""")

notebookutils.notebook.exit("SUCCESS")
